# 🟢 SwingBot — Colab Live Scanner
**Runs one scan manually. Sends Grade A & B signals to Telegram.**

### How to use
1. Fill in your `TELEGRAM_TOKEN` and `CHAT_ID` in **Cell 1 — Settings**
2. Run **Cell 2 — Universe + Sectors** (hardcoded — no upload needed)
3. Run **Cell 3 — Install & Imports** once per session
4. Run **Cell 4 — Core Functions** once per session
5. Run **Cell 8 — Pipeline Test** to confirm Telegram is working
6. Run **Cell 5 — Run Scanner** each time you want to scan (hourly during market hours)
7. Optionally run **Cell 6 — Heartbeat** at session start or **Cell 7 — Recap** at EOD

> **VIX rule**: Scanner is silent if VIX is outside 15–25. No override.
>
> **Stop/Target**: Stop = Entry − 1×ATR | Target = Entry + 2×ATR | R:R = 2:1
>
> **Dedup**: Each ticker alerts only once per session. Restart runtime to reset.
>
> **Universe**: 351 tickers hardcoded in Cell 2. Edit directly — no CSV upload needed.


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 1 — SETTINGS  ← fill these in before running
# ════════════════════════════════════════════════════════

TELEGRAM_TOKEN  = '8985416938:AAEgYmiSVegIc8Q9sOL5EHv3OGu2CjH5qI8'   # from @BotFather
CHAT_ID         = '8834650652'             # your Telegram chat ID

# ── Trade parameters (match your trading plan) ──────────
PRICE_MIN       = 5.0
PRICE_MAX       = 25.0
VIX_MIN         = 15.0
VIX_MAX         = 25.0
RISK_PER_TRADE  = 7.0
ATR_STOP_MULT   = 1.0
ATR_TARGET_MULT = 2.0
ATR_PERIOD      = 14
EARNINGS_DAYS   = 7

# ── Signal quality filters ──────────────────────────────
# MIN_RVOL retired — rvol reported in alert, judged in Trade Review (not a scanner kill)
# MAX_SWING_DELTA retired — swing delta reported in alert, judged in Trade Review (not a scanner kill)
MIN_FLOAT_M     = 10.0   # Minimum float in millions


# ── Time-adjusted RVol thresholds ───────────────────────
# RVol means different things at different times of day
# Earlier in the day = lower threshold acceptable
RVOL_THRESHOLDS = {
    '09:30': 0.3,   # First 30 min — 0.3x is strong
    '10:00': 0.5,   # 10:00-11:00 AM — 0.5x is on pace
    '11:00': 0.7,   # 11:00 AM-1:00 PM — 0.7x acceptable
    '13:00': 1.0,   # 1:00-2:00 PM — 1.0x needed
    '14:00': 1.2,   # After 2:00 PM — 1.2x needed
}

print('✅ Settings loaded.')

In [ ]:
# ==============================================================
# CELL 2 -- UNIVERSE + SECTORS
# Edit UNIVERSE list and SECTOR_MAP dict directly here.
# No file uploads needed.
# ==============================================================

# -- Your full ticker universe (edit freely) ---------------------------
UNIVERSE = [
    'ET', 'F', 'PCG', 'VG', 'KEY', 'VTRS', 'PR', 'PAA', 'KIM', 'PSLV',
    'HST', 'DOC', 'AUR', 'FHN', 'WULF', 'AM', 'AES', 'LUMN', 'RIOT', 'ONB',
    'AAL', 'CIFR', 'PRMB', 'LEVI', 'RIG', 'IBRX', 'VLY', 'NOV', 'CWAN', 'HR',
    'SGHC', 'S', 'FNB', 'GTES', 'OSCR', 'MAC', 'FLG', 'ZGN', 'M', 'BGC',
    'MARA', 'SBRA', 'GBTG', 'ZETA', 'PTEN', 'MDU', 'EBC', 'RELY', 'ONDS', 'WSC',
    'TFSL', 'CRGY', 'LION', 'COLD', 'KC', 'CLSK', 'FULT', 'LBTYA', 'DNP', 'IRT',
    'AMRX', 'FLNC', 'BNL', 'NVST', 'ARX', 'FBP', 'HIMX', 'CVBF', 'RUM', 'INFQ',
    'BTDR', 'OGN', 'ERAS', 'RDW', 'GNW', 'APLE', 'KYIV', 'CC', 'BKD', 'GEO',
    'SFNC', 'MPT', 'TXG', 'NN', 'DNLI', 'DBRG', 'RCUS', 'AKR', 'TNGX', 'WT',
    'BANC', 'PFS', 'DYN', 'LFST', 'IHS', 'UE', 'ALMS', 'DHT', 'QUBT', 'CALY',
    'SPNT', 'WTTR', 'UNIT', 'TALO', 'FA', 'VNET', 'BW', 'RLAY', 'FSLY', 'HUN',
    'HOG', 'DFTX', 'IMNM', 'OSW', 'AESI', 'MNR', 'HE', 'UA', 'PK', 'SOC',
    'AMPX', 'XIFR', 'PENN', 'TE', 'NEXT', 'POET', 'DRH', 'SLDE', 'ACHC', 'CXW',
    'DHC', 'BCRX', 'CTOS', 'NVCR', 'PUMP', 'PBI', 'SEM', 'PCT', 'NWBI', 'FUN',
    'TRVI', 'NTST', 'FLYW', 'LWLG', 'SHO', 'GNL', 'AUPH', 'PGNY', 'NEOG', 'NEXA',
    'FCF', 'JBLU', 'AGRO', 'ARCO', 'NAC', 'NRIX', 'XPRO', 'TH', 'VRE', 'KRP',
    'SFL', 'SNDX', 'MD', 'FIVN', 'BORR', 'EFC', 'PEB', 'SHLS', 'INVA', 'SGML',
    'HLIT', 'BDJ', 'NVRI', 'STGW', 'CMPS', 'GLUE', 'ABCL', 'HOPE', 'TSHA', 'SATL',
    'OPRA', 'XHR', 'VSTS', 'VIR', 'KW', 'DCH', 'NVAX', 'ECVT', 'AMLX', 'LAR',
    'HLX', 'TRIN', 'RLJ', 'ARI', 'SLS', 'BCAX', 'AHCO', 'CSWC', 'TTI', 'HCSG',
    'MRTN', 'LIND', 'CRML', 'TWO', 'NPKI', 'ACDC', 'CSIQ', 'PRA', 'ADTN', 'MUX',
    'FTRE', 'UAMY', 'TROX', 'GILT', 'IART', 'SG', 'TK', 'AVNS', 'NAT', 'CDNA',
    'DEC', 'HTLD', 'CIM', 'JBIO', 'GPRE', 'OCFC', 'HYLN', 'LPTH', 'CRVS', 'SVRA',
    'PDM', 'KURA', 'HPK', 'CFFN', 'KODK', 'KOPN', 'LXU', 'AVR', 'NRGV', 'EVC',
    'PSNL', 'DRTS', 'TALK', 'ANNX', 'ABX', 'CODI', 'AVTX', 'ARKO', 'MGTX', 'SPIR',
    'CGEM', 'OMER', 'CRSR', 'WEST', 'UMAC', 'ADAM', 'ABSI', 'AURA', 'BVS', 'ASC',
    'RLMD', 'CGNT', 'GRNT', 'MITK', 'GRPN', 'SXC', 'CYRX', 'DC', 'VELO', 'DSGN',
    'INN', 'AHRT', 'BLMN', 'BZH', 'SLDB', 'CLYM', 'GPRK', 'ALOY', 'EGY', 'ZVRA',
    'RYAM', 'CADL', 'INV', 'PACK', 'OIS', 'SRTA', 'KYTX', 'PANL', 'CRNC', 'OSPN',
    'PUBM', 'ASPN', 'IMMX', 'STTK', 'BLZE', 'IMXI', 'LZM', 'CAL', 'OSS', 'SGMT',
    'OPTX', 'VUZI', 'CCRN', 'ALMU', 'SIDU', 'GRRR', 'EOLS', 'MEI', 'CEPT', 'RILY',
    'AIRS', 'PAYS', 'XPER', 'QUIK', 'SHMD', 'DUOT', 'DBI', 'ATOM', 'GSIT', 'TENX',
    'RMAX', 'ABEO', 'BRUN', 'FNKO', 'LTRX', 'IMPP', 'CGEH', 'UNCY', 'MX', 'ARTV',
    'TORO', 'TRT', 'SKLZ', 'QMCO', 'AMPG', 'BNAI', 'PSIG', 'CPSH', 'PIII', 'EDSA',
    'CPIX', 'CUE', 'CLNN', 'IPWR', 'RDAC', 'ROLR', 'ASTI', 'FABC', 'AKTX', 'BTX',
    'BCAT',
]

# Deduplicate + normalize
UNIVERSE = [str(t).strip().upper().replace('.', '-') for t in UNIVERSE
            if str(t).strip().replace('-','').isalpha()]
UNIVERSE = list(dict.fromkeys(UNIVERSE))
print(f'\u2705 Universe: {len(UNIVERSE)} tickers')

# -- Sector map (all 351 tickers mapped) ------------------------------
SECTOR_MAP = {
    # Biotech
    'BCRX': 'Healthcare',
    'AURA': 'Biotech', 'BVS': 'Biotech', 'CGEH': 'Biotech', 'IMPP': 'Biotech', 'TXG': 'Biotech', 'UMAC': 'Biotech',
    # Closed-End Fund
    'DCH': 'Closed-End Fund', 'NAC': 'Closed-End Fund',
    # Commodities
    'PSLV': 'Commodities',
    # Consumer
    'AAL': 'Consumer', 'AGRO': 'Consumer', 'ARCO': 'Consumer', 'BLMN': 'Consumer', 'BRUN': 'Consumer', 'BZH': 'Consumer', 'CRSR': 'Consumer', 'DFTX': 'Consumer',
    'F': 'Consumer', 'FNKO': 'Consumer', 'FUN': 'Consumer', 'GRPN': 'Consumer', 'HOG': 'Consumer', 'INN': 'Consumer', 'JBLU': 'Consumer', 'LEVI': 'Consumer',
    'M': 'Consumer', 'OSCR': 'Consumer', 'OSW': 'Consumer', 'PENN': 'Consumer', 'SG': 'Consumer', 'SKLZ': 'Consumer', 'TH': 'Consumer', 'UA': 'Consumer',
    'ZGN': 'Consumer',
    # Crypto
    'ACDC': 'Crypto', 'BNAI': 'Crypto', 'BTDR': 'Crypto', 'CIFR': 'Crypto', 'CLSK': 'Crypto', 'MARA': 'Crypto', 'RIOT': 'Crypto', 'WULF': 'Crypto',
    # Energy
    'AM': 'Energy', 'AMPX': 'Energy', 'ARX': 'Energy', 'BORR': 'Energy', 'CRGY': 'Energy', 'CSIQ': 'Energy', 'DEC': 'Energy', 'DHT': 'Energy',
    'DYN': 'Energy', 'EGY': 'Energy', 'ET': 'Energy', 'GPRE': 'Energy', 'GPRK': 'Energy', 'HLX': 'Energy', 'HPK': 'Energy', 'KRP': 'Energy',
    'NAT': 'Energy', 'NOV': 'Energy', 'OIS': 'Energy', 'PAA': 'Energy', 'PTEN': 'Energy', 'PUMP': 'Energy', 'RIG': 'Energy', 'SFL': 'Energy',
    'TALO': 'Energy', 'TK': 'Energy', 'TTI': 'Energy', 'WTTR': 'Energy',
    # Financials
    'ARI': 'Financials', 'ARKO': 'Financials', 'BANC': 'Financials', 'BGC': 'Financials', 'CFFN': 'Financials', 'CIM': 'Financials', 'CODI': 'Financials', 'CSWC': 'Financials',
    'CVBF': 'Financials', 'DBRG': 'Financials', 'DC': 'Financials', 'EBC': 'Financials', 'EFC': 'Financials', 'FA': 'Financials', 'FBP': 'Financials', 'FCF': 'Financials',
    'FHN': 'Financials', 'FLG': 'Financials', 'FNB': 'Financials', 'FULT': 'Financials', 'GNW': 'Financials', 'HOPE': 'Financials', 'INV': 'Financials', 'KEY': 'Financials',
    'LIND': 'Financials', 'LION': 'Financials', 'NWBI': 'Financials', 'OCFC': 'Financials', 'ONB': 'Financials', 'PFS': 'Financials', 'PRA': 'Financials', 'RELY': 'Financials',
    'RILY': 'Financials', 'SFNC': 'Financials', 'TFSL': 'Financials', 'TRIN': 'Financials', 'TWO': 'Financials', 'VLY': 'Financials', 'WEST': 'Financials', 'WT': 'Financials',
    # Healthcare
    'ABCL': 'Healthcare', 'ABEO': 'Healthcare', 'ABSI': 'Healthcare', 'ACHC': 'Healthcare', 'AHCO': 'Healthcare', 'AHRT': 'Healthcare', 'AKTX': 'Healthcare', 'ALMS': 'Healthcare',
    'AMLX': 'Healthcare', 'AMRX': 'Healthcare', 'ANNX': 'Healthcare', 'ARTV': 'Healthcare', 'AUPH': 'Healthcare', 'AVNS': 'Healthcare', 'AVTX': 'Healthcare', 'BCAT': 'Healthcare',
    'BCAX': 'Healthcare', 'BKD': 'Healthcare', 'BTX': 'Healthcare', 'CADL': 'Healthcare', 'CALY': 'Healthcare', 'CCRN': 'Healthcare', 'CDNA': 'Healthcare', 'CEPT': 'Healthcare',
    'CGEM': 'Healthcare', 'CLNN': 'Healthcare', 'CLYM': 'Healthcare', 'CMPS': 'Healthcare', 'CPIX': 'Healthcare', 'CRVS': 'Healthcare', 'CUE': 'Healthcare', 'CYRX': 'Healthcare',
    'DHC': 'Healthcare', 'DNLI': 'Healthcare', 'EDSA': 'Healthcare', 'EOLS': 'Healthcare', 'ERAS': 'Healthcare', 'GLUE': 'Healthcare', 'HCSG': 'Healthcare', 'IBRX': 'Healthcare',
    'IMMX': 'Healthcare', 'IMNM': 'Healthcare', 'INVA': 'Healthcare', 'JBIO': 'Healthcare', 'KURA': 'Healthcare', 'KYTX': 'Healthcare', 'LFST': 'Healthcare', 'LWLG': 'Healthcare',
    'MD': 'Healthcare', 'MGTX': 'Healthcare', 'NEOG': 'Healthcare', 'NRIX': 'Healthcare', 'NVAX': 'Healthcare', 'NVCR': 'Healthcare', 'NVST': 'Healthcare', 'OGN': 'Healthcare',
    'OMER': 'Healthcare', 'OPTX': 'Healthcare', 'PGNY': 'Healthcare', 'PRMB': 'Healthcare', 'PSNL': 'Healthcare', 'RCUS': 'Healthcare', 'RLAY': 'Healthcare', 'RLMD': 'Healthcare',
    'SEM': 'Healthcare', 'SLDB': 'Healthcare', 'SLS': 'Healthcare', 'SNDX': 'Healthcare', 'SPNT': 'Healthcare', 'SRTA': 'Healthcare', 'STTK': 'Healthcare', 'SVRA': 'Healthcare',
    'TENX': 'Healthcare', 'TNGX': 'Healthcare', 'TRVI': 'Healthcare', 'TSHA': 'Healthcare', 'UNCY': 'Healthcare', 'VIR': 'Healthcare', 'ZVRA': 'Healthcare',
    # Industrials
    'AESI': 'Industrials', 'AIRS': 'Industrials', 'AMPG': 'Industrials', 'ASC': 'Industrials', 'ASPN': 'Industrials', 'BW': 'Industrials', 'CAL': 'Industrials', 'CC': 'Industrials',
    'CTOS': 'Industrials', 'DBI': 'Industrials', 'GRNT': 'Industrials', 'GTES': 'Industrials', 'HTLD': 'Industrials', 'HUN': 'Industrials', 'KALU': 'Industrials', 'LXU': 'Industrials',
    'MDU': 'Industrials', 'MEI': 'Industrials', 'MRTN': 'Industrials', 'MUX': 'Industrials', 'NEXA': 'Industrials', 'NN': 'Industrials', 'NVRI': 'Industrials', 'OSS': 'Industrials',
    'PACK': 'Industrials', 'PANL': 'Industrials', 'PBI': 'Industrials', 'PCT': 'Industrials', 'RYAM': 'Industrials', 'SGML': 'Industrials', 'SGMT': 'Industrials', 'SHLS': 'Industrials',
    'SIDU': 'Industrials', 'SXC': 'Industrials', 'TORO': 'Industrials', 'TROX': 'Industrials', 'TRT': 'Industrials', 'UAMY': 'Industrials', 'WSC': 'Industrials', 'XPRO': 'Industrials',
    # Materials
    'ABX': 'Materials', 'AVR': 'Materials', 'CRML': 'Materials', 'ECVT': 'Materials', 'GILT': 'Materials', 'IART': 'Materials', 'KODK': 'Materials', 'LZM': 'Materials',
    'NPKI': 'Materials', 'PR': 'Materials', 'VTRS': 'Materials',
    # REITs
    'AKR': 'REITs', 'APLE': 'REITs', 'BDJ': 'REITs', 'BNL': 'REITs', 'COLD': 'REITs', 'DNP': 'REITs', 'DOC': 'REITs', 'DRH': 'REITs',
    'GNL': 'REITs', 'HR': 'REITs', 'HST': 'REITs', 'IRT': 'REITs', 'KIM': 'REITs', 'KW': 'REITs', 'MAC': 'REITs', 'MNR': 'REITs',
    'MPT': 'REITs', 'NTST': 'REITs', 'PDM': 'REITs', 'PEB': 'REITs', 'PK': 'REITs', 'RLJ': 'REITs', 'RMAX': 'REITs', 'SBRA': 'REITs',
    'SHO': 'REITs', 'UE': 'REITs', 'VRE': 'REITs', 'XHR': 'REITs',
    # Special
    'KYIV': 'Special', 'LAR': 'Special', 'XIFR': 'Special',
    # Technology
    'ADAM': 'Technology', 'ADTN': 'Technology', 'ALMU': 'Technology', 'ALOY': 'Technology', 'ASTI': 'Technology', 'ATOM': 'Technology', 'AUR': 'Technology', 'BLZE': 'Technology',
    'CGNT': 'Technology', 'CPSH': 'Technology', 'CRNC': 'Technology', 'CWAN': 'Technology', 'DSGN': 'Technology', 'DUOT': 'Technology', 'FABC': 'Technology', 'FIVN': 'Technology',
    'FLYW': 'Technology', 'FSLY': 'Technology', 'FTRE': 'Technology', 'GBTG': 'Technology', 'GRRR': 'Technology', 'GSIT': 'Technology', 'HIMX': 'Technology', 'HLIT': 'Technology',
    'HYLN': 'Technology', 'IMXI': 'Technology', 'INFQ': 'Technology', 'IPWR': 'Technology', 'KC': 'Technology', 'KOPN': 'Technology', 'LBTYA': 'Technology', 'LPTH': 'Technology',
    'LTRX': 'Technology', 'MITK': 'Technology', 'MX': 'Technology', 'ONDS': 'Technology', 'OPRA': 'Technology', 'OSPN': 'Technology', 'PAYS': 'Technology', 'PIII': 'Technology',
    'POET': 'Technology', 'PSIG': 'Technology', 'PUBM': 'Technology', 'QMCO': 'Technology', 'QUBT': 'Technology', 'QUIK': 'Technology', 'RDAC': 'Technology', 'RDW': 'Technology',
    'ROLR': 'Technology', 'S': 'Technology', 'SATL': 'Technology', 'SGHC': 'Technology', 'SHMD': 'Technology', 'SLDE': 'Technology', 'SOC': 'Technology', 'SPIR': 'Technology',
    'STGW': 'Technology', 'VELO': 'Technology', 'VSTS': 'Technology', 'VUZI': 'Technology', 'XPER': 'Technology', 'ZETA': 'Technology',
    # Telecom
    'DRTS': 'Telecom', 'IHS': 'Telecom', 'RUM': 'Telecom', 'TALK': 'Telecom', 'VG': 'Telecom', 'VNET': 'Telecom',
    # Utilities
    'AES': 'Utilities', 'CXW': 'Utilities', 'EVC': 'Utilities', 'FLNC': 'Utilities', 'GEO': 'Utilities', 'HE': 'Utilities', 'LUMN': 'Utilities', 'NEXT': 'Utilities',
    'NRGV': 'Utilities', 'PCG': 'Utilities', 'TE': 'Utilities', 'UNIT': 'Utilities',

    # Additional tickers identified May 29, 2026
    'SWBI': 'Consumer',       # Smith & Wesson
    'JMSB': 'Financials',     # John Marshall Bank
    'GUT' : 'CEF',            # Gabelli Utility Trust
    'AOD' : 'CEF',            # Aberdeen Total Dynamic Dividend
    'WT'  : 'Financials',     # WisdomTree Investments
    'FFIC': 'Financials',     # Flushing Financial
    'NMR' : 'Financials',     # Nomura Holdings
    'KRP' : 'Energy',         # Kimbell Royalty Partners
    'LILA': 'Telecom',        # Liberty Latin America
    'SVRA': 'Healthcare',     # Savara Inc
    'REPL': 'Healthcare',     # Replimune Group
    'GHRS': 'Healthcare',     # GH Research
    'BDIMF': 'Materials',     # Black Diamond Group
    'OXLCN': 'Financials',    # Oxford Lane Capital
    'SBFG': 'Financials',     # SB Financial Group
    'BCX' : 'Materials',      # BlackRock Resources
    'GJT' : 'Financials',     # Strats SM Trust
}
print(f'\u2705 Sectors mapped: {len(SECTOR_MAP)}')


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 3 — INSTALL & IMPORTS  (run once per session)
# ════════════════════════════════════════════════════════

import subprocess
subprocess.run(['pip', 'install', 'yfinance==0.2.66', 'pandas', 'numpy', 'pytz', '-q'])

import json, time, urllib.request, warnings
import pandas as pd
from datetime import datetime, date, timedelta
import pytz
import numpy as np
import yfinance as yf
warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)

EST = pytz.timezone('America/New_York')

# Session-level dedup — resets when you restart runtime
ALERTED_TODAY = {}

# ── US Market Holidays 2026 ──────────────────────────────
MARKET_HOLIDAYS = {
    date(2026, 1, 1), date(2026, 1, 19), date(2026, 2, 16),
    date(2026, 4, 3), date(2026, 5, 25), date(2026, 7, 3),
    date(2026, 9, 7), date(2026, 11, 26), date(2026, 12, 25),
}

print('✅ Imports done. yfinance:', yf.__version__)

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 4 — CORE FUNCTIONS  (run once per session)
# ════════════════════════════════════════════════════════

# ── Telegram ────────────────────────────────────────────
def send_telegram(text):
    url  = f'https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage'
    data = json.dumps({'chat_id': CHAT_ID, 'text': text, 'parse_mode': 'HTML'}).encode()
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, data=data,
                                         headers={'Content-Type': 'application/json'})
            with urllib.request.urlopen(req, timeout=10) as r:
                result = json.loads(r.read())
                if result.get('ok'):
                    return True
        except Exception as e:
            if attempt < 2: time.sleep(2)
    print(f'  ⚠️  Telegram send failed after 3 attempts')
    return False

# ── Helpers ──────────────────────────────────────────────
def get_est_time():
    return datetime.now(EST)

def fetch_with_retry(func, retries=3, delay=5):
    for attempt in range(retries):
        try:
            result = func()
            if result is not None and not (hasattr(result, 'empty') and result.empty):
                return result
        except Exception:
            pass
        if attempt < retries - 1:
            time.sleep(delay)
    return None

# ── VIX ──────────────────────────────────────────────────
def get_vix():
    try:
        def fetch():
            v = yf.download('^VIX', period='5d', interval='1d',
                            progress=False, auto_adjust=True)
            if v.empty: return None
            if v.index.tz is not None: v.index = v.index.tz_localize(None)
            return float(v['Close'].dropna().iloc[-1])
        return fetch_with_retry(fetch)
    except:
        return None

# ── Daily close + SMA200 ─────────────────────────────────
def get_daily_close(tickers):
    try:
        end   = date.today()
        start = end - timedelta(days=300)
        raw   = fetch_with_retry(lambda: yf.download(
            tickers, start=start.strftime('%Y-%m-%d'),
            end=end.strftime('%Y-%m-%d'),
            interval='1d', progress=False, auto_adjust=True
        ))
        if raw is None or raw.empty: return {}, {}
        if raw.index.tz is not None:
            raw.index = raw.index.tz_convert('America/New_York').tz_localize(None)
        close_df = raw['Close']
        # Handle single ticker (flat columns)
        if isinstance(close_df, pd.Series):
            close_df = close_df.to_frame(name=tickers[0])
        prices, sma200s = {}, {}
        for ticker in tickers:
            if ticker not in close_df.columns: continue
            s = close_df[ticker].dropna()
            if s.empty: continue
            prices[ticker]  = float(s.iloc[-1])
            sma = s.rolling(200, min_periods=50).mean().dropna()
            if not sma.empty: sma200s[ticker] = float(sma.iloc[-1])
        return prices, sma200s
    except:
        return {}, {}

# ── 1H data ──────────────────────────────────────────────
def get_hourly(tickers):
    try:
        end   = date.today() + timedelta(days=1)
        start = date.today() - timedelta(days=5)
        raw   = fetch_with_retry(lambda: yf.download(
            tickers, start=start.strftime('%Y-%m-%d'),
            end=end.strftime('%Y-%m-%d'),
            interval='1h', progress=False, auto_adjust=True
        ))
        if raw is None or raw.empty: return None
        if raw.index.tz is not None:
            raw.index = raw.index.tz_convert('America/New_York').tz_localize(None)
        return raw
    except:
        return None

# ── Earnings check ───────────────────────────────────────
def check_earnings(ticker):
    try:
        t   = yf.Ticker(ticker)
        cal = t.calendar
        if cal is None or (hasattr(cal, 'empty') and cal.empty): return False
        if isinstance(cal, pd.DataFrame):
            if 'Earnings Date' in cal.index:
                earn = pd.Timestamp(cal.loc['Earnings Date'].iloc[0])
            elif 'Earnings Date' in cal.columns:
                earn = pd.Timestamp(cal['Earnings Date'].iloc[0])
            else: return False
            earn = earn.tz_localize(None) if earn.tzinfo else earn
            diff = (earn - pd.Timestamp(date.today())).days
            return 0 <= diff <= EARNINGS_DAYS
    except:
        pass
    return False

# ── Daily bars for grading ───────────────────────────────
def get_bars(ticker, daily_raw):
    try:
        close_df  = daily_raw['Close']
        open_df   = daily_raw['Open']
        high_df   = daily_raw['High']
        low_df    = daily_raw['Low']
        volume_df = daily_raw['Volume']
        if ticker not in close_df.columns: return None
        df = pd.DataFrame({
            'open'  : open_df[ticker].dropna(),
            'high'  : high_df[ticker].dropna(),
            'low'   : low_df[ticker].dropna(),
            'close' : close_df[ticker].dropna(),
            'volume': volume_df[ticker].dropna(),
        }).tail(60)
        return df if len(df) >= 5 else None
    except:
        return None

# ── Grading (10-question checklist) ─────────────────────
def grade_signal(bars, price, vix):
    if bars is None:
        return 2, 'D', [1,1,0,0,0,0,0,0,0,0]
    scores = []
    # Q1 — EMA cross confirmed (auto-pass — scanner confirmed it)
    scores.append(1)
    # Q2 — Above 200 SMA (auto-pass — scanner confirmed it)
    scores.append(1)
    # Q3 — Price above 10/20 EMA last 3 bars
    try:
        e10 = bars['close'].ewm(span=10, adjust=False).mean()
        e20 = bars['close'].ewm(span=20, adjust=False).mean()
        c3  = bars['close'].iloc[-3:].values
        scores.append(1 if all(c3 > e10.iloc[-3:].values) and
                          all(c3 > e20.iloc[-3:].values) else 0)
    except: scores.append(0)
    # Q4 — ADX > 20
    try:
        if len(bars) >= 20:
            h,l,c = bars['high'], bars['low'], bars['close']
            tr  = pd.concat([h-l,(h-c.shift(1)).abs(),(l-c.shift(1)).abs()],axis=1).max(axis=1)
            up  = h.diff(); dn = -l.diff()
            pdm = up.where((up>dn)&(up>0), 0.0)
            ndm = dn.where((dn>up)&(dn>0), 0.0)
            a14 = tr.rolling(14).mean()
            pdi = 100*pdm.rolling(14).mean()/a14
            ndi = 100*ndm.rolling(14).mean()/a14
            dx  = 100*(pdi-ndi).abs()/(pdi+ndi)
            adx = dx.rolling(14).mean()
            scores.append(1 if float(adx.iloc[-1]) > 20 else 0)
        else: scores.append(0)
    except: scores.append(0)
    # Q5 — Near swing high/low (within 3%)
    try:
        sh = bars.tail(20)['high'].max()
        sl = bars.tail(20)['low'].min()
        scores.append(1 if (abs(price-sh)/price<=0.03 or
                            abs(price-sl)/price<=0.03) else 0)
    except: scores.append(0)
    # Q6 — VIX in range
    scores.append(1 if vix and VIX_MIN <= float(vix) <= VIX_MAX else 0)
    # Q7 — Strong bullish candle
    try:
        last = bars.iloc[-1]
        body = abs(last['close']-last['open'])
        rng  = last['high']-last['low']
        scores.append(1 if rng>0 and body/rng>=0.5 and last['close']>last['open'] else 0)
    except: scores.append(0)
    # Q8 — Daily EMA9 > EMA21
    try:
        e9  = bars['close'].ewm(span=9,  adjust=False).mean()
        e21 = bars['close'].ewm(span=21, adjust=False).mean()
        scores.append(1 if float(e9.iloc[-1]) > float(e21.iloc[-1]) else 0)
    except: scores.append(0)
    # Q9 — Clear stop placement
    try:
        if len(bars) >= 14:
            tr  = pd.concat([bars['high']-bars['low'],
                             (bars['high']-bars['close'].shift(1)).abs(),
                             (bars['low'] -bars['close'].shift(1)).abs()],axis=1).max(axis=1)
            atr = float(tr.rolling(14).mean().iloc[-1])
            sl  = float(bars['low'].tail(10).min())
            scores.append(1 if (price-2*atr) <= sl <= price else 0)
        else: scores.append(0)
    except: scores.append(0)
    # Q10 — Volume >= 1.5x 20-bar avg
    try:
        avg = bars['volume'].tail(21).iloc[:-1].mean()
        scores.append(1 if avg>0 and float(bars['volume'].iloc[-1])>=1.5*avg else 0)
    except: scores.append(0)

    total = sum(scores)
    grade = 'A' if total>=9 else 'B' if total>=7 else 'C' if total>=5 else 'D'
    return total, grade, scores

# ── ATR ──────────────────────────────────────────────────
def calc_atr(bars):
    try:
        if bars is None or len(bars) < ATR_PERIOD: return None
        tr = pd.concat([
            bars['high']-bars['low'],
            (bars['high']-bars['close'].shift(1)).abs(),
            (bars['low'] -bars['close'].shift(1)).abs()
        ], axis=1).max(axis=1)
        return float(tr.rolling(ATR_PERIOD).mean().iloc[-1])
    except:
        return None

# ── EMA cross detection ──────────────────────────────────
def has_cross(series):
    ema9  = series.ewm(span=9,  adjust=False).mean()
    ema21 = series.ewm(span=21, adjust=False).mean()
    fast  = (ema9 > ema21).astype(bool)
    prev  = fast.shift(1).fillna(False).astype(bool)
    return fast & ~prev


# -- Relative Volume at signal time ------------------------------------
def get_rvol(bars, hourly_raw, ticker):
    try:
        if hourly_raw is not None:
            vol_df = hourly_raw['Volume']
            if isinstance(vol_df, pd.Series):
                vol_df = vol_df.to_frame(name=ticker)
            if ticker in vol_df.columns:
                today_str  = pd.Timestamp(date.today()).strftime('%Y-%m-%d')
                today_bars = vol_df[ticker].dropna()
                today_bars = today_bars[today_bars.index.strftime('%Y-%m-%d') == today_str]
                today_vol  = float(today_bars.sum()) if not today_bars.empty else None
                if today_vol and today_vol > 0:
                    avg_vol = float(bars['volume'].tail(20).mean()) if bars is not None else 0
                    if avg_vol > 0:
                        return round(today_vol / avg_vol, 2)
        if bars is not None and len(bars) >= 2:
            last_vol = float(bars['volume'].iloc[-1])
            avg_vol  = float(bars['volume'].tail(21).iloc[:-1].mean())
            if avg_vol > 0:
                return round(last_vol / avg_vol, 2)
    except:
        pass
    return None

# -- Distance from recent swing high (%) --------------------------------
def get_dist_from_swing_high(bars, price):
    try:
        if bars is None or len(bars) < 5:
            return None
        swing_high = float(bars.tail(20)['high'].max())
        if swing_high <= 0:
            return None
        return round((swing_high - price) / swing_high * 100, 1)
    except:
        return None

# -- Float / Market Cap -------------------------------------------------
def get_float_mktcap(ticker):
    try:
        info  = yf.Ticker(ticker).info or {}
        flt   = info.get('floatShares') or info.get('impliedSharesOutstanding')
        mkt   = info.get('marketCap')
        return (round(flt/1e6,1) if flt else None,
                round(mkt/1e6,1) if mkt else None)
    except:
        return None, None

# -- Format alert (extended) -------------------------------------------
def format_alert(signals, scan_time, data_time):
    now_str  = scan_time.strftime('%I:%M %p EST')
    if data_time:
        data_str  = data_time.strftime('%I:%M %p EST')
        lag_mins  = int((scan_time.replace(tzinfo=None) - data_time.replace(tzinfo=None) if hasattr(data_time,'tzinfo') else scan_time - data_time).total_seconds() / 60)
        lag_str   = f' ({lag_mins}m old)' if lag_mins > 15 else ''
        data_line = f'Data as of: {data_str}{lag_str}'
    else:
        data_line = 'Data as of: Unknown'
    lines = [
        f'\U0001f7e2 <b>SIGNAL ALERT \u2014 {now_str}</b>',
        data_line,
        '\u2501'*21
    ]
    for s in signals:
        rvol    = s.get('rvol')
        dist_sh = s.get('dist_swing_high')
        flt     = s.get('float_m')
        mkt     = s.get('mktcap_m')
        lines += [
            f'<b>{s["ticker"]}</b> | Grade {s["grade"]} | Score {s["score"]}/10',
            f'Entry   : ${s["entry"]:.2f}',
            f'Stop    : ${s["stop"]:.2f}',
            f'Target  : ${s["target"]:.2f}',
            f'ATR     : ${s["atr"]:.3f}',
            f'Shares  : {s["shares"]}',
            f'Sector  : {s.get("sector", "N/A")}',
            f'Rel Vol : {get_rvol_context(rvol, scan_time) if rvol is not None else "N/A"}',
            f'Swing \u0394 : {str(dist_sh)+"% below swing high" if dist_sh is not None else "N/A"}',
            f'Float   : {str(flt)+"M" if flt is not None else "N/A"}'  
            f'  |  Mkt Cap: {"$"+str(mkt)+"M" if mkt is not None else "N/A"}',
            f'VIX     : {s["vix"]}',
            f'Earnings: {s["earnings"]}',
            '\u2501'*21
        ]
    lines.append(f'{len(signals)} signal{"s" if len(signals)>1 else ""} found')
    return '\n'.join(lines)


# ── Time-adjusted RVol threshold ─────────────────────────
def get_rvol_threshold(signal_time=None):
    """
    Returns the minimum acceptable RVol based on time of day.
    Earlier signals need lower RVol — volume hasn't had time to build.
    """
    if signal_time is None:
        signal_time = get_est_time()

    hour   = signal_time.hour
    minute = signal_time.minute
    t      = hour * 100 + minute  # e.g. 10:30 AM = 1030

    if   t < 1000: return 0.3   # Before 10:00 AM
    elif t < 1100: return 0.5   # 10:00–11:00 AM
    elif t < 1300: return 0.7   # 11:00 AM–1:00 PM
    elif t < 1400: return 1.0   # 1:00–2:00 PM
    else:          return 1.2   # After 2:00 PM

def get_rvol_context(rvol, signal_time=None):
    """
    Returns a human-readable RVol assessment based on time of day.
    Used in alert formatting and pre-screen output.
    """
    if rvol is None:
        return "N/A"
    threshold = get_rvol_threshold(signal_time)
    if   rvol >= 1.5:              return f"{rvol}x 🟢 Strong"
    elif rvol >= threshold * 1.2:  return f"{rvol}x 🟢 On pace"
    elif rvol >= threshold:        return f"{rvol}x 🟡 Acceptable for this time"
    else:                          return f"{rvol}x 🔴 Weak for this time"

print('\u2705 All functions loaded.')


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 5 — RUN SCANNER  ← run this each time you want to scan
# ════════════════════════════════════════════════════════

def run_scan():
    global ALERTED_TODAY

    now = get_est_time()
    print(f'\n🔍 SwingBot scan started — {now.strftime("%I:%M %p EST")} on {now.strftime("%A, %b %d")}')

    # ── Holiday check ────────────────────────────────────
    if date.today() in MARKET_HOLIDAYS:
        print('🏖️  Market holiday — scan skipped.')
        return

    # ── VIX check ────────────────────────────────────────
    print('  Fetching VIX...', end=' ')
    vix = get_vix()
    vix_str = str(round(vix, 2)) if vix else 'N/A'
    print(vix_str)

    if vix and (vix < VIX_MIN or vix > VIX_MAX):
        print(f'⛔  VIX {vix_str} outside 15–25. No trades. Scanner silent.')
        return

    # ── Dedup — reset if new day ─────────────────────────
    if ALERTED_TODAY.get('_date') != str(date.today()):
        ALERTED_TODAY = {'_date': str(date.today())}
        print('  📅 New day — dedup reset.')

    # ── Daily data ───────────────────────────────────────
    # Universe is pre-filtered for $5-$25 and above SMA200
    # We still fetch daily prices to confirm current price is in range
    print(f'  Fetching daily prices for {len(UNIVERSE)} tickers...', end=' ')
    prices, sma200s = get_daily_close(UNIVERSE)
    print(f'{len(prices)} returned.')

    # Only filter out tickers where data fetch failed or price drifted outside range
    candidates = [
        t for t in UNIVERSE
        if t in prices and PRICE_MIN <= prices[t] <= PRICE_MAX
    ]
    print(f'  Candidates after price range check: {len(candidates)}')

    if not candidates:
        print('  No candidates — data fetch may have failed.')
        return

    # ── 1H data ──────────────────────────────────────────
    print('  Fetching 1H data...', end=' ')
    hourly_raw = get_hourly(candidates)
    if hourly_raw is None:
        print('FAILED — could not fetch 1H data.')
        return
    hourly_close = hourly_raw['Close']
    if isinstance(hourly_close, pd.Series):
        hourly_close = hourly_close.to_frame(name=candidates[0])
    print('done.')

    # ── Daily bars for grading ───────────────────────────
    end_d   = date.today()
    start_d = end_d - timedelta(days=300)
    daily_raw = fetch_with_retry(lambda: yf.download(
        candidates,
        start=start_d.strftime('%Y-%m-%d'), end=end_d.strftime('%Y-%m-%d'),
        interval='1d', progress=False, auto_adjust=True
    ))
    if daily_raw is not None and daily_raw.index.tz is not None:
        daily_raw.index = daily_raw.index.tz_localize(None)

    # ── Last data timestamp — show today's latest bar only ─────────────
    data_time = None
    try:
        today_str  = str(date.today())
        idx        = hourly_close.index
        # Index is already EST-naive after tz_convert in get_hourly
        idx_naive  = idx.tz_localize(None) if idx.tz is not None else idx
        today_mask = idx_naive.strftime('%Y-%m-%d') == today_str
        if today_mask.any():
            last_ts = idx_naive[today_mask][-1]
            data_time = last_ts.to_pydatetime()
            # Warn if last bar is stale (> 2 hours old)
            staleness = (now.replace(tzinfo=None) - data_time).total_seconds() / 3600
            if staleness > 2:
                print(f'  ⚠️  Stale data — last bar: {data_time.strftime("%I:%M %p")} ({staleness:.1f}h ago). Run Cell 3 to refresh.')
        else:
            last_ts   = idx_naive[-1]
            data_time = last_ts.to_pydatetime()
            print(f'  ⚠️  No data for today — using {data_time.strftime("%b %d %I:%M %p")}. yfinance may be delayed.')
    except:
        pass

    # ── Cross detection window ───────────────────────────
    # Scan from market open (9:30 AM EST) to now — full session
    market_open = now.replace(hour=9, minute=30, second=0, microsecond=0)
    scan_window_start = market_open
    signals_found     = []

    # Diagnostics — track why tickers are skipped
    skip_no_col = skip_short = skip_no_cross = skip_prefilter = skip_grade = skip_atr = 0

    # Normalize scan window to naive timestamp for comparison
    scan_ts = pd.Timestamp(market_open.strftime('%Y-%m-%d %H:%M:%S'))

    print(f'  Scanning for EMA crosses since market open ({market_open.strftime("%I:%M %p EST")})...')
    for ticker in candidates:
        if ticker in ALERTED_TODAY:
            continue
        if ticker not in hourly_close.columns:
            skip_no_col += 1
            continue

        s = hourly_close[ticker].dropna()
        if len(s) < 10:
            skip_short += 1
            continue

        cross  = has_cross(s)
        # Normalize index to naive for comparison
        idx = cross.index
        if idx.tz is not None:
            idx = idx.tz_localize(None)
        window = cross.values[idx >= scan_ts]
        if not window.any():
            skip_no_cross += 1
            continue

        price = float(s.iloc[-1])
        if not (PRICE_MIN <= price <= PRICE_MAX): continue
        # Note: SMA200 check removed — universe is pre-filtered

        # ── Hard pre-filters (fast kills before expensive grading) ──────────
        bars = get_bars(ticker, daily_raw) if daily_raw is not None else None

        # ATR quality: must be >= 0.5% of price
        atr = calc_atr(bars)
        if atr is None or atr <= 0:
            skip_atr += 1
            continue
        rvol = get_rvol(bars, hourly_raw, ticker)

        dist_sh = get_dist_from_swing_high(bars, price)

        float_m, mktcap_m = get_float_mktcap(ticker)
        if float_m is not None and float_m < MIN_FLOAT_M:
            print(f'    ⛔ {ticker} — Float {float_m}M (min {MIN_FLOAT_M}M)')
            skip_atr += 1
            continue

        # ── Grade ───────────────────────────────────────────────────────────
        score, grade, _ = grade_signal(bars, price, vix)

        if grade not in ('A', 'B'):
            skip_grade += 1
            continue

        stop     = round(price - ATR_STOP_MULT   * atr, 2)
        target   = round(price + ATR_TARGET_MULT * atr, 2)
        shares   = max(1, int(RISK_PER_TRADE / (ATR_STOP_MULT * atr)))
        earnings = 'YES' if check_earnings(ticker) else 'NO'

        # -- enrichment fields (already computed above) ----------------
        # Sector lookup — SECTOR_MAP first, yfinance fallback for unknowns
        sector = SECTOR_MAP.get(ticker.upper(), 'Unknown')
        if sector == 'Unknown':
            try:
                info   = yf.Ticker(ticker).info or {}
                sector = info.get('sector') or info.get('industryDisp') or 'Unknown'
                SECTOR_MAP[ticker.upper()] = sector
            except:
                sector = 'Unknown'
        # rvol, dist_sh, float_m, mktcap_m already set in pre-filter block

        signals_found.append({
            'ticker'          : ticker,
            'grade'           : grade,
            'score'           : score,
            'entry'           : round(price, 2),
            'stop'            : stop,
            'target'          : target,
            'atr'             : round(atr, 3),
            'shares'          : shares,
            'vix'             : round(vix, 2) if vix else 'N/A',
            'earnings'        : earnings,
            'sector'          : sector,
            'rvol'            : rvol,
            'dist_swing_high' : dist_sh,
            'float_m'         : float_m,
            'mktcap_m'        : mktcap_m,
        })
        ALERTED_TODAY[ticker] = {
            'time'  : now.strftime('%I:%M %p'),
            'grade' : grade,
            'entry' : round(price, 2)
        }
        rvol_ctx = get_rvol_context(rvol, now)
        print(f'    ✅ SIGNAL: {ticker} | Grade {grade} | Score {score}/10 | ${round(price,2)} | RVol: {rvol_ctx}')

    # ── Send alert ───────────────────────────────────────
    if signals_found:
        msg = format_alert(signals_found, now, data_time)
        sent = send_telegram(msg)
        print(f'\n📲 Alert sent to Telegram: {"✅" if sent else "❌ Failed"}')
        print(f'   {len(signals_found)} signal{"s" if len(signals_found)>1 else ""} — {[s["ticker"] for s in signals_found]}')
    else:
        print(f'\n💤 No new Grade A/B signals this scan.')
        print(f'   Skipped: {skip_no_col} no 1H data | {skip_short} too few bars | {skip_no_cross} no cross | {skip_atr} bad ATR/float | {skip_grade} grade C/D')

    print(f'\nDone. Alerted this session: {[k for k in ALERTED_TODAY if not k.startswith("_")]}')

# ── RUN ──────────────────────────────────────────────────
run_scan()

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 5.5 — TOS WATCHLIST SCAN (Google Sheets driven)
# ════════════════════════════════════════════════════════
#
# DAILY WORKFLOW:
# 1. TOS scanner fires on your phone
# 2. Screenshot → send to Claude in chat
# 3. Claude parses tickers → gives you a list
# 4. Open Google Sheet → Scanner tab → paste tickers
#    one per row starting at A2 (A1 = "Ticker" header)
# 5. Run this cell — reads Sheet, grades all tickers,
#    fires full alert to Telegram
#
# SHEET SETUP (one time only):
# - Sheet: SwingBot Trade Journal
# - Tab name: scanner
# - Column A header (A1): Ticker
# - Tickers go in A2, A3, A4... one per row
# - Sheet must be shared: Anyone with link can view
# ════════════════════════════════════════════════════════

# ── Google Sheet config ──────────────────────────────────
SHEET_ID   = "1wxgzlC4NnjOf42RaD-I67IFX61VhRDPoGe5UNZMcnRE"
TAB_NAME   = "scanner"

# ════════════════════════════════════════════════════════
# NO CHANGES NEEDED BELOW THIS LINE
# ════════════════════════════════════════════════════════

def read_tickers_from_sheet(sheet_id, tab_name):
    """
    Reads tickers from Column A of the specified Google Sheet tab.
    No API key required — uses public CSV export endpoint.
    Sheet must be shared as Anyone with link can view.
    """
    import urllib.request
    from io import StringIO

    url = (
        f"https://docs.google.com/spreadsheets/d/{sheet_id}"
        f"/gviz/tq?tqx=out:csv&sheet={tab_name}"
    )
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            content = r.read().decode("utf-8")
        df = pd.read_csv(StringIO(content))
        # Get first column, skip header, drop blanks
        tickers = (
            df.iloc[:, 0]
            .dropna()
            .astype(str)
            .str.strip()
            .str.upper()
            .replace("", pd.NA)
            .dropna()
            .tolist()
        )
        # Remove header row if it accidentally got included
        tickers = [t for t in tickers if t not in ("TICKER", "SYMBOL", "")]
        # Deduplicate preserving order
        tickers = list(dict.fromkeys(tickers))
        return tickers
    except Exception as e:
        print(f"  ❌ Could not read Google Sheet: {e}")
        print(f"  URL tried: {url}")
        print(f"  Check: Sheet is shared as Anyone with link can view")
        return []


def run_tos_scan():
    """
    Full SwingBot grading on tickers read from Google Sheet.
    Same grading logic as run_scan() — skips universe loop.
    """
    global ALERTED_TODAY

    now = get_est_time()
    print(f'\n📱 TOS Watchlist Scan — {now.strftime("%I:%M %p EST")} on {now.strftime("%A, %b %d")}')

    # ── Read tickers from Google Sheet ───────────────────
    print(f'  Reading tickers from Google Sheet (tab: {TAB_NAME})...', end=' ')
    tickers = read_tickers_from_sheet(SHEET_ID, TAB_NAME)

    if not tickers:
        print('\n  ❌ No tickers found in Sheet.')
        print(f'  Make sure tickers are in Column A starting at row 2')
        print(f'  Sheet tab must be named: {TAB_NAME}')
        return

    print(f'{len(tickers)} tickers found.')
    print(f'  Tickers: {tickers}')
    print('═' * 50)

    # ── VIX check ────────────────────────────────────────
    print('  Fetching VIX...', end=' ')
    vix = get_vix()
    vix_str = str(round(vix, 2)) if vix else 'N/A'
    print(vix_str)

    if vix and (vix < VIX_MIN or vix > VIX_MAX):
        print(f'⛔ VIX {vix_str} outside {VIX_MIN}–{VIX_MAX}. No trades today.')
        return

    # ── Dedup reset if new day ───────────────────────────
    from datetime import date
    if ALERTED_TODAY.get('_date') != str(date.today()):
        ALERTED_TODAY = {'_date': str(date.today())}
        print('  📅 New day — dedup reset.')

    # ── Daily price data ─────────────────────────────────
    print(f'  Fetching daily prices...', end=' ')
    prices, sma200s = get_daily_close(tickers)
    print(f'{len(prices)} returned.')

    if not prices:
        print('  ❌ No price data returned — check tickers in Sheet')
        return

    # ── 1H data ──────────────────────────────────────────
    print('  Fetching 1H data...', end=' ')
    hourly_raw = get_hourly(tickers)
    if hourly_raw is None:
        print('FAILED.')
        return
    hourly_close = hourly_raw['Close']
    if isinstance(hourly_close, pd.Series):
        hourly_close = hourly_close.to_frame(name=tickers[0])
    print('done.')

    # ── Daily bars for grading ───────────────────────────
    from datetime import timedelta
    end_d   = date.today()
    start_d = end_d - timedelta(days=300)
    daily_raw = fetch_with_retry(lambda: yf.download(
        tickers,
        start=start_d.strftime('%Y-%m-%d'),
        end=end_d.strftime('%Y-%m-%d'),
        interval='1d', progress=False, auto_adjust=True
    ))
    if daily_raw is not None and daily_raw.index.tz is not None:
        daily_raw.index = daily_raw.index.tz_localize(None)

    # ── Last data timestamp ──────────────────────────────
    data_time = None
    try:
        today_str  = str(date.today())
        idx        = hourly_close.index
        idx_naive  = idx.tz_localize(None) if idx.tz is not None else idx
        today_mask = idx_naive.strftime('%Y-%m-%d') == today_str
        if today_mask.any():
            data_time = idx_naive[today_mask][-1].to_pydatetime()
    except:
        pass

    # ── Grade each ticker ────────────────────────────────
    signals_found = []
    skipped       = []

    for ticker in tickers:
        print(f'\n  [{ticker}]')

        # Price check
        if ticker not in prices:
            print(f'    ⚠️  No price data — skip')
            skipped.append(f'{ticker}: no price data')
            continue

        price = prices[ticker]
        if not (PRICE_MIN <= price <= PRICE_MAX):
            print(f'    ⚠️  Price ${price:.2f} outside ${PRICE_MIN}–${PRICE_MAX} — skip')
            skipped.append(f'{ticker}: price ${price:.2f} out of range')
            continue

        # SMA200 check
        sma200 = sma200s.get(ticker)
        if sma200 and price < sma200:
            print(f'    ⚠️  Below SMA200 (${sma200:.2f}) — skip')
            skipped.append(f'{ticker}: below SMA200')
            continue

        # Daily bars
        bars = get_bars(ticker, daily_raw) if daily_raw is not None else None

        # ATR
        atr = calc_atr(bars)
        if atr is None or atr <= 0:
            print(f'    ⚠️  ATR unavailable — skip')
            skipped.append(f'{ticker}: no ATR')
            continue

        # ATR% check
        atr_pct = atr / price * 100
        if atr_pct < 0.5:
            print(f'    ⛔ ATR {round(atr_pct,2)}% too tight — skip')
            skipped.append(f'{ticker}: ATR {round(atr_pct,2)}% too tight')
            continue

        # Float check
        float_m, mktcap_m = get_float_mktcap(ticker)
        if float_m is not None and float_m < MIN_FLOAT_M:
            print(f'    ⛔ Float {float_m}M too small — skip')
            skipped.append(f'{ticker}: float {float_m}M too small')
            continue

        # Relative volume + swing delta
        rvol    = get_rvol(bars, hourly_raw, ticker)
        dist_sh = get_dist_from_swing_high(bars, price)

        # Swing delta check
        if dist_sh is not None and dist_sh > 5.0:
            print(f'    ⛔ Swing Δ {dist_sh}% > 5% — skip')
            skipped.append(f'{ticker}: swing delta {dist_sh}% too wide')
            continue

        # Grade
        score, grade, scores = grade_signal(bars, price, vix)
        rvol_threshold = get_rvol_threshold(now)
        rvol_ctx       = get_rvol_context(rvol, now)
        print(f'    Score: {score}/10 | Grade: {grade} | RVol: {rvol_ctx} (min {rvol_threshold}x at this time) | ATR%: {round(atr_pct,2)}%')

        if grade not in ('A', 'B'):
            print(f'    ⛔ Grade {grade} — skip')
            skipped.append(f'{ticker}: grade {grade}')
            continue

        # Earnings check
        earnings = 'YES' if check_earnings(ticker) else 'NO'
        if earnings == 'YES':
            print(f'    ⛔ Earnings within 7 days — skip')
            skipped.append(f'{ticker}: earnings within 7 days')
            continue

        # Position sizing
        stop   = round(price - ATR_STOP_MULT   * atr, 2)
        target = round(price + ATR_TARGET_MULT * atr, 2)
        shares = max(1, int(RISK_PER_TRADE / (ATR_STOP_MULT * atr)))
        # Sector lookup — SECTOR_MAP first, yfinance fallback for unknowns
        sector = SECTOR_MAP.get(ticker, 'Unknown')
        if sector == 'Unknown':
            try:
                info   = yf.Ticker(ticker).info or {}
                sector = info.get('sector') or info.get('industryDisp') or 'Unknown'
                # Cache it for this session
                SECTOR_MAP[ticker] = sector
                print(f'    📍 Sector auto-detected: {sector}')
            except:
                sector = 'Unknown'

        signals_found.append({
            'ticker'         : ticker,
            'grade'          : grade,
            'score'          : score,
            'entry'          : round(price, 2),
            'stop'           : stop,
            'target'         : target,
            'atr'            : round(atr, 3),
            'shares'         : shares,
            'vix'            : round(vix, 2) if vix else 'N/A',
            'earnings'       : earnings,
            'sector'         : sector,
            'rvol'           : rvol,
            'dist_swing_high': dist_sh,
            'float_m'        : float_m,
            'mktcap_m'       : mktcap_m,
        })

        ALERTED_TODAY[ticker] = {
            'time' : now.strftime('%I:%M %p'),
            'grade': grade,
            'entry': round(price, 2)
        }
        print(f'    ✅ SIGNAL: {ticker} | Grade {grade} | Score {score}/10 | ${round(price,2)} | RVol: {rvol_ctx}')

    # ── Summary ──────────────────────────────────────────
    print(f'\n{"═"*50}')
    print(f'  Checked  : {len(tickers)} tickers')
    print(f'  Signals  : {len(signals_found)}')
    print(f'  Skipped  : {len(skipped)}')
    rvol_threshold = get_rvol_threshold(now)
    print(f'  RVol min at scan time ({now.strftime("%I:%M %p")}): {rvol_threshold}x')
    for s in skipped:
        print(f'    ⛔ {s}')

    # ── Send to Telegram ─────────────────────────────────
    if signals_found:
        msg  = format_alert(signals_found, now, data_time)
        sent = send_telegram(msg)
        print(f'\n📲 Telegram alert: {"✅ Sent" if sent else "❌ Failed"}')
        print(f'   Signals: {[s["ticker"] for s in signals_found]}')
    else:
        print(f'\n💤 No qualifying signals from TOS list.')

    print(f'\nDone.')

# ── RUN ──────────────────────────────────────────────────
run_tos_scan()


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 6 — SEND HEARTBEAT  (optional — test Telegram connection)
# ════════════════════════════════════════════════════════

def send_heartbeat():
    vix = get_vix()
    vix_str  = str(round(vix, 2)) if vix else 'N/A'
    vix_note = ''
    if vix:
        if   vix < VIX_MIN: vix_note = f' ⚠️ Below {VIX_MIN} — no trades today'
        elif vix > VIX_MAX: vix_note = f' ⚠️ Above {VIX_MAX} — no trades today'
        else:                vix_note = ' ✅ In range'
    msg = (f'✅ <b>SwingBot Online — Colab</b>\n'
           f'Manual scan session started\n'
           f'━━━━━━━━━━━━━━━━━━━━━\n'
           f'Universe : {len(UNIVERSE)} tickers\n'
           f'VIX      : {vix_str}{vix_note}\n'
           f'Scans    : run Cell 5 each hour (9:30AM–4:00PM EST)')
    sent = send_telegram(msg)
    print(f'Heartbeat sent: {"✅" if sent else "❌ Failed"}  |  VIX: {vix_str}')

send_heartbeat()

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 7 — SEND DAILY RECAP  (optional — run at end of day)
# ════════════════════════════════════════════════════════

def send_recap():
    vix     = get_vix()
    vix_str = str(round(vix, 2)) if vix else 'N/A'
    now_str = get_est_time().strftime('%B %d, %Y')
    # Remove internal _date key
    alerted = {k:v for k,v in ALERTED_TODAY.items() if not k.startswith('_')}

    if not alerted:
        msg = (f'📊 <b>SwingBot Daily Recap — {now_str}</b>\n'
               f'━━━━━━━━━━━━━━━━━━━━━\n'
               f'No signals today\n'
               f'VIX: {vix_str}')
    else:
        lines = [
            f'📊 <b>SwingBot Daily Recap — {now_str}</b>',
            f'━━━━━━━━━━━━━━━━━━━━━',
            f'Signals today: {len(alerted)}',
            '━━━━━━━━━━━━━━━━━━━━━'
        ]
        for ticker, info in alerted.items():
            lines.append(f'{info["time"]} — <b>{ticker}</b> | Grade {info["grade"]} | Entry ${info["entry"]}')
        lines += ['━━━━━━━━━━━━━━━━━━━━━', f'VIX today: {vix_str}']
        msg = '\n'.join(lines)

    sent = send_telegram(msg)
    print(f'Recap sent: {"✅" if sent else "❌ Failed"}')
    print(f'Signals alerted: {list(alerted.keys()) if alerted else "None"}')

send_recap()

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 8 — PIPELINE TEST  (optional — confirm Telegram works)
# ════════════════════════════════════════════════════════

def test_pipeline():
    vix     = get_vix()
    now_str = get_est_time().strftime('%I:%M %p EST')
    msg = (f'\U0001f7e2 <b>SwingBot Pipeline Test — Colab</b>\n'
           f'━━━━━━━━━━━━━━━━━━━━━\n'
           f'Colab → Telegram: ✅ Working\n'
           f'Time     : {now_str}\n'
           f'VIX      : {round(vix,2) if vix else "N/A"}\n'
           f'Universe : {len(UNIVERSE)} tickers\n'
           f'━━━━━━━━━━━━━━━━━━━━━\n'
           f'SwingBot Colab scanner is ready!')
    sent = send_telegram(msg)
    print(f'Test message sent: {"✅" if sent else "❌ Check your TELEGRAM_TOKEN and CHAT_ID"}')

test_pipeline()